# 기존 M5 설계 빠른 확인: 단일 음성 BPR, Dunnhumby seed 42

M1·M2·M4·M5 네 모형만 seed 42로 새로 학습합니다. 기존 10시드 설계에서 K=5 음성 평균을 양성 1개–균등 음성 1개의 BPR 비교로 바꾸고, 나머지 구조·분할·100 epoch·하이퍼파라미터는 유지합니다. 총 학습은 4회입니다. 이미 노출된 test의 단일시드 방향성 확인이므로 이 결과로 추가 튜닝하거나 최종 결론을 내리지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, os, shutil, subprocess, sys

REVIEWED_SHA = 'f9da379ebc5744f9ab1ff5a372214e8c4ea03fa6'
REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
os.chdir('/content')
repo = Path('/content/clv-m2-lightgcn-runner')
clone_errors = []
for clone_attempt in range(1, 4):
    if repo.exists():
        shutil.rmtree(repo)
    result = subprocess.run(['git', 'clone', REPO_URL, str(repo)], text=True, capture_output=True)
    if result.returncode == 0:
        break
    clone_errors.append(result.stderr.strip())
    print(f'GitHub clone {clone_attempt}/3 실패:', result.stderr.strip())
else:
    raise RuntimeError('GitHub clone 3회 실패:\n' + '\n'.join(clone_errors))
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for module_name in tuple(sys.modules):
    if module_name.startswith(('lightgcn_', 'clv_')):
        del sys.modules[module_name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('실행 코드 고정 완료:', actual_sha)


In [ ]:
import json
import torch
from lightgcn_clv_m5_nv_economic_positive_weight_k1_test import (
    configure_m5_nv_economic_positive_k1_seed42_run,
    preflight_summary,
    run_m5_nv_economic_positive_k1_test,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_m5_nv_economic_positive_k1_seed42_run(
    out_dir='/content/drive/MyDrive/논문/data/results_v3_dunnhumby_m5_explicit_nv_personalized_positive_weight_single_negative_bpr_test_seed42_v1',
)
summary = preflight_summary(cfg)
assert cfg.seeds == (42,)
assert cfg.negative_count == 1
assert len(summary['models']) == 4
assert summary['unchanged_from_prior_ten_seed_run']['validation_constructed'] is False
assert summary['unchanged_from_prior_ten_seed_run']['holdout_constructed'] is False
assert summary['task_guards']['new_item_task'] is True
assert 'single-seed' in summary['protocol_status']
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = run_m5_nv_economic_positive_k1_test(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

print('1) seed 42 M1·M2·M4·M5 전체 절대지표')
show(result_df)
print('2) M1·M4 기준 전체 지표 비교')
show(result_df.attrs['comparison'])
print('3) M2 × M4 상호작용 — 기술적 보조 결과')
show(result_df.attrs['interaction'])
print('4) 단일시드 기술적 판독')
print(json.dumps(result_df.attrs['descriptive_reading'], ensure_ascii=False, indent=2))
print('5) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
